In [27]:
import glob
import pandas as pd

# Load data

In [28]:
def load_evidence(fn):
    ds_name = fn.split("/")[-3]
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    extracted_df = df['extracted'].apply(pd.Series).add_prefix('extracted_')
    # derived_df = df['derived'].apply(pd.Series).add_prefix('derived_')
    # Combine the original DataFrame with the unwrapped columns
    # df = pd.concat([df.drop(columns=['source', 'extracted', 'derived']), source_df, extracted_df, derived_df], axis=1)
    df = pd.concat([df.drop(columns=['source', 'extracted']), source_df, extracted_df], axis=1)
    df["ds"] = ds_name
    del df["derived"]
    return df

In [29]:
fns = glob.glob("../../data/*/evidence_llm*/evidence.json")


In [30]:
dfs = []
for fn in fns:
    dfs.append(load_evidence(fn))
df = pd.concat(dfs)

# Perform token alignment

In [31]:
tdf = df.query("source_type == 'text'").copy()

In [32]:
tdf.head(2)

,source_type,source_rationale,source_id,extracted_organism,extracted_cell_type_label,extracted_cell_source,extracted_cell_state,extracted_feature_name,extracted_feature_type,ds
0,text,Monocyte 1 strongly expressed FCGR3A (CD16) an...,text,homo_sapiens,Monocyte,None,None,FCGR3A,gene,kidney_Wu2018
1,text,"Interestingly, ABCA1, which mediates sterol ef...",text,homo_sapiens,Monocyte,None,None,ABCA1,gene,kidney_Wu2018


In [33]:
from extract.extract import align_ng, norm_text, reconstruct_target_by_token

In [34]:
# find the cell type
tdf["found_aln_cell_type_label"] = tdf.apply(
    lambda x: align_ng(x["source_rationale"], x["extracted_cell_type_label"])[0], axis=1
)
# reconstruct the cell type
tdf["source_cell_type_label"] = tdf["found_aln_cell_type_label"].apply(lambda x: reconstruct_target_by_token("", x[0] if len(x)>0 else ""))

# find the feature name
tdf["found_aln_feature_name"] = tdf.apply(
    lambda x: align_ng(x["source_rationale"], x["extracted_feature_name"])[0], axis=1
)
# reconstruct the feature name
tdf["source_feature_name"] = tdf["found_aln_feature_name"].apply(lambda x: reconstruct_target_by_token("", x[0] if len(x)>0 else ""))



# tdf["source_cell_type_label"] = tdf.apply(
#     lambda x: align_ng(norm_text(x["source_rationale"]), norm_text(x["extracted_cell_type_label"]))[0], 
#     axis=1)
# tdf["source_feature_name"]    = tdf.apply(
#     lambda x: align_ng(norm_text(x["source_rationale"]), norm_text(x["extracted_feature_name"]))[0], 
#     axis=1)

In [35]:
tdf["found_feature_name"] = tdf["source_feature_name"].apply(norm_text) == tdf["extracted_feature_name"].apply(norm_text)
tdf["found_cell_type_label"] = tdf["source_cell_type_label"].apply(norm_text) == tdf["extracted_cell_type_label"].apply(norm_text)

In [36]:
agg = tdf.groupby("ds").agg({
    "found_feature_name": ["count", "sum"], 
    "found_cell_type_label": ["count", "sum"], 
    }).rename(columns={"count": "total", "sum": "found"}, level=1)
agg[("found_feature_name", "pct")] = 100*agg[("found_feature_name", "found")] / agg[("found_feature_name", "total")]
agg[("found_cell_type_label", "pct")] = 100*agg[("found_cell_type_label", "found")] / agg[("found_cell_type_label", "total")]
agg[("found_feature_name", "complete")] = agg[("found_feature_name", "pct")] == 100
agg[("found_cell_type_label", "complete")] = agg[("found_cell_type_label", "pct")] == 100
agg["all_complete"] = agg[("found_feature_name", "complete")] & agg[("found_cell_type_label", "complete")]

In [37]:
# Add a total row at the bottom of the DataFrame
total_row = agg.sum(numeric_only=True)
total_row[("found_feature_name", "pct")] = 100 * total_row[("found_feature_name", "found")] / total_row[("found_feature_name", "total")]
total_row[("found_cell_type_label", "pct")] = 100 * total_row[("found_cell_type_label", "found")] / total_row[("found_cell_type_label", "total")]

# Handling boolean columns for completeness
total_row[("found_feature_name", "complete")] = total_row[("found_feature_name", "pct")] == 100
total_row[("found_cell_type_label", "complete")] = total_row[("found_cell_type_label", "pct")] == 100
total_row["all_complete"] = total_row[("found_feature_name", "complete")] & total_row[("found_cell_type_label", "complete")]

# Add the total row to the DataFrame
agg.loc["total"] = total_row

# Sort the columns
agg = agg.sort_index(axis=1)

agg.reset_index()

/var/folders/89/1b_bcy3s39g4nkyxbdxvgzt40000gn/T/ipykernel_92642/1931357756.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  total_row[("found_feature_name", "complete")] = total_row[("found_feature_name", "pct")] == 100


ds all_complete found_cell_type_label         \
                                                       complete  found   
0          adipose_Emont2022        False                 False    5.0   
1       adipose_Hildreth2021        False                 False   26.0   
2         adipose_Jaitin2019        False                 False    0.0   
3        adipose_Massier2023        False                 False    0.0   
4        adipose_Merrick2019        False                 False   11.0   
5          adipose_Vijay2019        False                 False    9.0   
6             bladder_Yu2019        False                 False    7.0   
7                bone_He2021        False                 False    8.0   
8      cortex_Booeshaghi2021        False                 False    0.0   
9             eye_Gautam2021        False                 False    0.0   
10          heart_Tucker2020        False                 False    5.0   
11             kidney_Wu2018        False                 False    2.0   
12        liver_Guillams2022        False                 False    0.0   
13            lung_Adams2020        False                 False    0.0   
14          ovary_Wagner2020        False                 False    6.0   
15  pancreas_Segerstolpe2016        False                 False   13.0   
16          placenta_Liu2018        False                 False    4.0   
17         testis_Shamis2020        False                 False   17.0   
18           yolksac_Goh2023        False                 False    2.0   
19                     total        False                 False  115.0   

                      found_feature_name                            
          pct   total           complete  found        pct   total  
0   27.777778    18.0              False    0.0   0.000000    18.0  
1   27.083333    96.0              False   14.0  14.583333    96.0  
2    0.000000    50.0              False    1.0   2.000000    50.0  
3    0.000000    34.0              False   15.0  44.117647    34.0  
4   45.833333    24.0              False    4.0  16.666667    24.0  
5   12.500000    72.0              False    6.0   8.333333    72.0  
6   13.725490    51.0              False    3.0   5.882353    51.0  
7   11.267606    71.0              False    3.0   4.225352    71.0  
8    0.000000     6.0              False    0.0   0.000000     6.0  
9    0.000000    24.0              False    3.0  12.500000    24.0  
10   7.352941    68.0              False   18.0  26.470588    68.0  
11   6.451613    31.0              False    8.0  25.806452    31.0  
12   0.000000     3.0              False    0.0   0.000000     3.0  
13   0.000000    41.0              False    6.0  14.634146    41.0  
14  10.000000    60.0              False   13.0  21.666667    60.0  
15  10.569106   123.0              False    7.0   5.691057   123.0  
16   4.395604    91.0              False    9.0   9.890110    91.0  
17  12.500000   136.0              False   19.0  13.970588   136.0  
18   1.769912   113.0              False   36.0  31.858407   113.0  
19  10.341727  1112.0              False  165.0  14.838129  1112.0

In [38]:
nobs = tdf.shape[0]
found_ct = tdf["found_cell_type_label"].sum()
found_fn = tdf["found_feature_name"].sum()
frac_ct = found_ct / nobs * 100
frac_fn = found_fn / nobs * 100


In [39]:
# find specific datasets with missing cell type or feature name
ds = 'adipose_Merrick2019'
bad = tdf.query(f"ds == '{ds}' & (~found_cell_type_label | ~found_feature_name)")[["source_rationale", "extracted_cell_type_label", "source_cell_type_label", "extracted_feature_name", "source_feature_name"]]
bad

,source_rationale,extracted_cell_type_label,source_cell_type_label,extracted_feature_name,source_feature_name
1,Adipose progenitor cells have been isolated on...,Adipose progenitor cells,Adipose progenitor cells,CD29,29
2,Adipose progenitor cells have been isolated on...,Adipose progenitor cells,Adipose progenitor cells,CD34,34
3,Adipose progenitor cells have been isolated on...,Adipose progenitor cells,Adipose progenitor cells,SCA1/LY6A,SCA16A
4,Adipose progenitor cells have been isolated on...,Adipose progenitor cells,Adipose progenitor cells,CD24,24
5,Preadipocyte factor–1 (PREF1) [also called del...,Adipocyte,ipocyte,PREF1,PREF1
6,Preadipocyte factor–1 (PREF1) [also called del...,Adipocyte,ipocyte,DLK1,DLK1
7,Preadipocyte factor–1 (PREF1) [also called del...,Adipocyte,ipocyte,PPARγ,PPARg
8,"Lastly, mural and smooth muscle–related cells ...",Smooth Muscle,,Acta2,a2
9,"Lastly, mural and smooth muscle–related cells ...",Smooth Muscle,,Myh11,h11
10,"Lastly, mural and smooth muscle–related cells ...",Smooth Muscle,,Pdgfrβ,dgfrb


In [22]:
bad.iloc[0]

source_rationale             Adipose progenitor cells have been isolated on...
extracted_cell_type_label                             Adipose progenitor cells
source_cell_type_label                                Adipose progenitor cells
extracted_feature_name                                                    CD29
source_feature_name                                                         29
Name: 1, dtype: object

In [23]:
bad.iloc[0].source_rationale


'Adipose progenitor cells have been isolated on the basis of their expression of common progenitor or mesenchymal cell surface markers, including platelet-derived growth factor receptor–α (PDGFRα), CD29, CD34, stem cell antigen–1 (SCA1)/LY6A, and CD24'

In [24]:
bad.iloc[0].extracted_feature_name



'CD29'

In [26]:
align_ng(bad.iloc[0].source_rationale, " CD29")[0]


[[{'token': ' CD', 'enc_token': 11325, 'start_idx': 196, 'end_idx': 199},
  {'token': '29', 'enc_token': 1682, 'start_idx': 199, 'end_idx': 201}]]